# Unausgeglichene Daten (Imbalanced Data)

Was tun wenn eine Klasse viel seltener ist als die andere?

## Inhaltsverzeichnis
1. Das Problem mit unausgeglichenen Daten
2. Das Problem sichtbar machen
3. Oversampling — Minderheitsklasse aufstocken
4. Undersampling — Mehrheitsklasse reduzieren
5. SMOTE — synthetische Datenpunkte erzeugen
6. Alle Methoden im Vergleich


## 1. Das Problem mit unausgeglichenen Daten

**Beispiel:** Diabetes-Datensatz
- 65% keine Diabetes (Klasse 0)
- 35% Diabetes (Klasse 1)

**Das Modell ohne Anpassung:**
- Lernt hauptsächlich die Mehrheitsklasse zu erkennen
- Accuracy klingt gut (~75%)
- Aber: Recall für Diabetes-Fälle ist schlecht!

**Warum gefährlich?**
```
Wenn das Modell 40% aller Diabetes-Fälle übersieht →
→ echte Menschen bekommen keine Diagnose
→ hohe Accuracy verschleiert das Problem
```

**Die Lösung:** Trainingsdaten ausbalancieren  
⚠️ Wichtig: **NUR die Trainingsdaten** ausbalancieren! Testdaten bleiben wie sie sind!


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils import resample
import seaborn as sns

# Diabetes-Datensatz laden (lokal oder von URL)
try:
    diabetes = pd.read_csv('data-imbalanced/diabetes.csv')
except:
    # Alternativ direkt laden
    url = "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"
    diabetes = pd.read_csv(url)

print(f"Datensatzgröße: {diabetes.shape}")
print()
print("Klassenverteilung:")
print(diabetes['Outcome'].value_counts())
print()
print("In Prozent:")
print((diabetes['Outcome'].value_counts(normalize=True) * 100).round(1))

In [ ]:
# Klassenverteilung visualisieren
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

diabetes['Outcome'].value_counts().plot(kind='bar', ax=axes[0], 
                                          color=['steelblue', 'salmon'],
                                          edgecolor='black')
axes[0].set_title("Absolut")
axes[0].set_xticklabels(['Keine Diabetes (0)', 'Diabetes (1)'], rotation=0)
axes[0].set_ylabel("Anzahl")

diabetes['Outcome'].value_counts(normalize=True).plot(kind='bar', ax=axes[1],
                                                         color=['steelblue', 'salmon'],
                                                         edgecolor='black')
axes[1].set_title("In Prozent")
axes[1].set_xticklabels(['Keine Diabetes (0)', 'Diabetes (1)'], rotation=0)
axes[1].set_ylabel("Anteil")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

plt.suptitle("Klassenverteilung im Diabetes-Datensatz", fontsize=13)
plt.tight_layout()
plt.show()

## 2. Baseline — Modell ohne Ausbalancierung

In [ ]:
X = diabetes.drop('Outcome', axis=1)
y = diabetes['Outcome']

# Skalierung (wichtig für Logistische Regression)
scaler = StandardScaler()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Baseline Modell
lr_baseline = LogisticRegression(max_iter=1000)
lr_baseline.fit(X_train_s, y_train)
pred_baseline = lr_baseline.predict(X_test_s)

print("=== BASELINE (ohne Ausbalancierung) ===")
print(f"Accuracy:  {lr_baseline.score(X_test_s, y_test):.4f}")
print(f"Precision: {precision_score(y_test, pred_baseline):.4f}")
print(f"Recall:    {recall_score(y_test, pred_baseline):.4f}  ← nur {recall_score(y_test, pred_baseline):.0%} der Diabetes-Fälle erkannt!")
print(f"F1-Score:  {f1_score(y_test, pred_baseline):.4f}")
print()
cm = confusion_matrix(y_test, pred_baseline)
print("Confusion Matrix:")
print(cm)
print(f"→ {cm[1,0]} von {cm[1,0]+cm[1,1]} echten Diabetes-Fällen übersehen!")

## 3. Oversampling — Minderheitsklasse aufstocken

**Idee:** Datenpunkte der Minderheitsklasse werden **zufällig dupliziert** bis beide Klassen gleich groß sind.

```
Vorher:  500 × Keine Diabetes, 268 × Diabetes
         ↓ resample(Diabetes, n=500, replace=True)
Nachher: 500 × Keine Diabetes, 500 × Diabetes ✅
```

**Vorteil:** Keine echten Daten verloren  
**Nachteil:** Keine neuen Informationen — nur Kopien

⚠️ **NUR auf Trainingsdaten anwenden!**


In [ ]:
# Trainingsdaten vorbereiten
train = pd.concat([X_train, y_train], axis=1)

# Klassen trennen
keine_diabetes = train[train['Outcome'] == 0]
ja_diabetes    = train[train['Outcome'] == 1]

print(f"Vor Oversampling: {len(keine_diabetes)} × Klasse 0, {len(ja_diabetes)} × Klasse 1")

# Minderheitsklasse auf Größe der Mehrheitsklasse hochsampeln
ja_diabetes_over = resample(ja_diabetes,
                             replace=True,           # MIT Zurücklegen (Duplikate erlaubt)
                             n_samples=len(keine_diabetes),
                             random_state=0)

print(f"Nach Oversampling: {len(keine_diabetes)} × Klasse 0, {len(ja_diabetes_over)} × Klasse 1")

# Neues Trainingsset zusammenbauen
train_over = pd.concat([keine_diabetes, ja_diabetes_over])
X_train_over = scaler.fit_transform(train_over.drop('Outcome', axis=1))
y_train_over = train_over['Outcome']

# Modell trainieren
lr_over = LogisticRegression(max_iter=1000)
lr_over.fit(X_train_over, y_train_over)
pred_over = lr_over.predict(X_test_s)

print()
print("=== NACH OVERSAMPLING ===")
print(f"Precision: {precision_score(y_test, pred_over):.4f}")
print(f"Recall:    {recall_score(y_test, pred_over):.4f}  ← Verbesserung!")
print(f"F1-Score:  {f1_score(y_test, pred_over):.4f}")

## 4. Undersampling — Mehrheitsklasse reduzieren

**Idee:** Datenpunkte der Mehrheitsklasse werden **zufällig entfernt** bis beide Klassen gleich groß sind.

```
Vorher:  500 × Keine Diabetes, 268 × Diabetes
         ↓ resample(Keine_Diabetes, n=268, replace=False)
Nachher: 268 × Keine Diabetes, 268 × Diabetes ✅
```

**Vorteil:** Keine Duplikate  
**Nachteil:** Datenverlust — wir werfen echte Informationen weg!

→ Nur sinnvoll wenn der Datensatz **groß genug** ist.


In [ ]:
# Mehrheitsklasse auf Größe der Minderheitsklasse reduzieren
keine_diabetes_under = resample(keine_diabetes,
                                 replace=False,          # OHNE Zurücklegen
                                 n_samples=len(ja_diabetes),
                                 random_state=0)

print(f"Nach Undersampling: {len(keine_diabetes_under)} × Klasse 0, {len(ja_diabetes)} × Klasse 1")

train_under = pd.concat([keine_diabetes_under, ja_diabetes])
X_train_under = scaler.fit_transform(train_under.drop('Outcome', axis=1))
y_train_under = train_under['Outcome']

lr_under = LogisticRegression(max_iter=1000)
lr_under.fit(X_train_under, y_train_under)
pred_under = lr_under.predict(X_test_s)

print()
print("=== NACH UNDERSAMPLING ===")
print(f"Precision: {precision_score(y_test, pred_under):.4f}")
print(f"Recall:    {recall_score(y_test, pred_under):.4f}")
print(f"F1-Score:  {f1_score(y_test, pred_under):.4f}")

## 5. SMOTE — Synthetische Datenpunkte erzeugen

**SMOTE** = Synthetic Minority Oversampling Technique

**Idee:** Statt echte Datenpunkte zu kopieren, werden **neue synthetische Datenpunkte** erzeugt!

**Wie?**
1. Nimm einen Datenpunkt der Minderheitsklasse
2. Finde seine k nächsten Nachbarn (auch Minderheitsklasse)
3. Erstelle einen neuen Punkt **zwischen** ihnen (zufällige Interpolation)

```
Echter Punkt A: [2.1, 3.4, 1.2]
Echter Punkt B: [2.8, 3.1, 1.5]
Neuer Punkt:   [2.4, 3.3, 1.3]  ← irgendwo dazwischen
```

**Vorteil:** Neue, diverse Datenpunkte — nicht nur Kopien!  
**Nachteil:** Kann gelegentlich unrealistische Datenpunkte erzeugen


In [ ]:
try:
    from imblearn.over_sampling import SMOTE

    sm = SMOTE(random_state=123, sampling_strategy=1.0)
    X_train_smote, y_train_smote = sm.fit_resample(X_train, y_train)

    print(f"Nach SMOTE: {sum(y_train_smote==0)} × Klasse 0, {sum(y_train_smote==1)} × Klasse 1")

    X_train_smote_s = scaler.fit_transform(X_train_smote)

    lr_smote = LogisticRegression(max_iter=1000)
    lr_smote.fit(X_train_smote_s, y_train_smote)
    pred_smote = lr_smote.predict(X_test_s)

    print()
    print("=== NACH SMOTE ===")
    print(f"Precision: {precision_score(y_test, pred_smote):.4f}")
    print(f"Recall:    {recall_score(y_test, pred_smote):.4f}")
    print(f"F1-Score:  {f1_score(y_test, pred_smote):.4f}")

except ImportError:
    print("imbalanced-learn installieren: pip install imbalanced-learn")

## 6. Alle Methoden im Vergleich

In [ ]:
# Großer Vergleich
ergebnisse = {
    'Baseline (unbalanciert)': pred_baseline,
    'Oversampling':            pred_over,
    'Undersampling':           pred_under,
}

try:
    ergebnisse['SMOTE'] = pred_smote
except:
    pass

print(f"{'Methode':<30} {'Precision':>10} {'Recall':>8} {'F1':>8}")
print("-" * 60)
for name, pred in ergebnisse.items():
    p = precision_score(y_test, pred)
    r = recall_score(y_test, pred)
    f = f1_score(y_test, pred)
    print(f"{name:<30} {p:>10.4f} {r:>8.4f} {f:>8.4f}")

print()
print("Fazit: Recall steigt durch alle Methoden — weniger Diabetes-Fälle übersehen!")
print("Preis: Precision sinkt etwas — mehr falsche Alarme. Das ist der Trade-off.")

## Zusammenfassung

### Wann welche Methode?

| Methode | Idee | Vorteil | Nachteil |
|---------|------|---------|---------|
| **Oversampling** | Minderheit duplizieren | Kein Datenverlust | Nur Kopien, kein neues Wissen |
| **Undersampling** | Mehrheit reduzieren | Keine Duplikate | Datenverlust |
| **SMOTE** | Synthetische Punkte | Neue Diversität | Kann unrealistisch sein |

### Die wichtigste Regel
```
⚠️ NUR Trainingsdaten ausbalancieren!
   Testdaten müssen die echte Welt widerspiegeln!
```

### Welche Metrik verwenden?
Bei unausgeglichenen Daten immer:
- **Recall** (wenn übersehen gefährlich)
- **F1-Score** (Kompromiss)
- **Balanced Accuracy**
- NIE nur Accuracy!
